# Cellpose-SAM Batch Segmentation

Runs Cellpose-SAM on ~6000 cell images stored in Google Drive and saves masks back to Drive.

**Before running:**
- Set runtime to GPU: Runtime → Change runtime type → T4 GPU (or A100 if available)
- Make sure Google Drive is mounted (cell 2)

**Resumable:** if Colab disconnects, just re-run from cell 4 onward — already-processed images are skipped automatically.

## 1. Check GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime → Change runtime type → T4 GPU and re-run."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

## 3. Install dependencies

In [ ]:
!pip install -q cellpose tifffile tqdm
import cellpose
print(f'cellpose version: {cellpose.__version__}')

## 4. Config

In [ ]:
from pathlib import Path

ROOT = Path('/content/drive/My Drive/Fusion AI/Prof Huang Project/Cellpose feature extractions')

# (input tiles folder, output masks folder) — one pair per stiffness condition
FOLDER_PAIRS = [
    (
        ROOT / 'imgs/260513_TC_Level/tiles',
        ROOT / 'masks/260513_TC_Level',
    ),
    (
        ROOT / 'imgs/260514_900kPa/tiles',
        ROOT / 'masks/260514_900kPa',
    ),
    (
        ROOT / 'imgs/260516_5kPa/tiles',
        ROOT / 'masks/260516_5kPa',
    ),
    (
        ROOT / 'imgs/260516_500kPa/tiles',
        ROOT / 'masks/260516_500kPa',
    ),
    (
        ROOT / 'imgs/260521_150kPa/tiles',
        ROOT / 'masks/260521_150kPa',
    ),
    (
        ROOT / 'imgs/260522_500kPa/tiles',
        ROOT / 'masks/260522_500kPa',
    ),
]

MODEL_TYPE  = 'cpsam'   # Cellpose-SAM
DIAMETER    = None       # None = auto-estimate per image
CHANNELS    = [0, 0]     # [0,0] = grayscale / use all channels for SAM
SKIP_EXISTING = True     # set False to reprocess everything from scratch
IMG_EXTS    = {'.tif', '.tiff', '.png', '.jpg', '.jpeg'}

print('Config:')
for in_dir, out_dir in FOLDER_PAIRS:
    n = len([p for p in in_dir.iterdir() if p.suffix.lower() in IMG_EXTS]) if in_dir.exists() else '?'
    print(f'  {in_dir.parent.name}  →  {n} images')

## 5. Load model (once — reused across all 6000 images)

In [ ]:
from cellpose import models

model = models.CellposeModel(gpu=True, model_type=MODEL_TYPE)
print(f'Model loaded: {MODEL_TYPE}  |  GPU={model.gpu}')

## 6. Segmentation functions

In [ ]:
import numpy as np
import tifffile
from tqdm.notebook import tqdm


def segment_one(img_path: Path, out_path: Path, model) -> None:
    """Read one image, run Cellpose-SAM, save uint16 label mask."""
    img = tifffile.imread(str(img_path))

    # Ensure shape is (H, W, 3) — Cellpose-SAM expects RGB
    if img.ndim == 2:
        img = np.stack([img, img, img], axis=-1)   # grayscale → RGB
    elif img.ndim == 3 and img.shape[0] in (1, 3):  # (C, H, W) → (H, W, C)
        img = np.moveaxis(img, 0, -1)
        if img.shape[-1] == 1:
            img = np.concatenate([img, img, img], axis=-1)

    masks, _, _ = model.eval(
        img,
        diameter=DIAMETER,
        channels=CHANNELS,
        normalize=True,
    )

    # Save as uint16 with LZW compression to keep Drive footprint small
    tifffile.imwrite(
        str(out_path),
        masks.astype(np.uint16),
        compression='lzw',
    )


def segment_folder(in_dir: Path, out_dir: Path, model) -> dict:
    """Process all images in in_dir, writing masks to out_dir."""
    out_dir.mkdir(parents=True, exist_ok=True)

    img_paths = sorted([p for p in in_dir.iterdir() if p.suffix.lower() in IMG_EXTS])
    if not img_paths:
        print(f'  [WARN] No images found in {in_dir}')
        return {'processed': 0, 'skipped': 0, 'failed': 0}

    processed = skipped = failed = 0
    failed_files = []

    for img_path in tqdm(img_paths, desc=in_dir.parent.name, unit='img'):
        out_path = out_dir / f'{img_path.stem}_masks.tif'

        if SKIP_EXISTING and out_path.exists():
            skipped += 1
            continue

        try:
            segment_one(img_path, out_path, model)
            processed += 1
        except Exception as e:
            failed += 1
            failed_files.append(img_path.name)
            print(f'  [FAIL] {img_path.name}: {e}')

    if failed_files:
        print(f'  Failed files: {failed_files}')

    return {'processed': processed, 'skipped': skipped, 'failed': failed}


print('Functions defined.')

## 7. Run segmentation on all folders

Expected time: ~10–30 min per 1000 images on T4 GPU.  
If Colab disconnects, re-run this cell — already-processed images are skipped.

In [ ]:
import time

grand_total = {'processed': 0, 'skipped': 0, 'failed': 0}
t_start = time.time()

for in_dir, out_dir in FOLDER_PAIRS:
    print(f'\n── {in_dir.parent.name} ──')
    if not in_dir.exists():
        print(f'  [SKIP] Input folder not found: {in_dir}')
        continue

    t0 = time.time()
    stats = segment_folder(in_dir, out_dir, model)
    elapsed = time.time() - t0

    print(f'  processed={stats["processed"]}  skipped={stats["skipped"]}  '
          f'failed={stats["failed"]}  ({elapsed/60:.1f} min)')

    for k in grand_total:
        grand_total[k] += stats[k]

total_elapsed = time.time() - t_start
print(f'\n══ DONE ══')
print(f'Total processed : {grand_total["processed"]}')
print(f'Total skipped   : {grand_total["skipped"]}')
print(f'Total failed    : {grand_total["failed"]}')
print(f'Wall time       : {total_elapsed/60:.1f} min')

## 8. Sanity check — compare one mask to website output

Optional: visually verify that the programmatic masks match what you got from cellpose.org.
Set `SAMPLE_IMAGE` to any image path you already processed manually.

In [ ]:
import matplotlib.pyplot as plt
import tifffile
from pathlib import Path

# Change these two paths to a real image and its manually-downloaded mask
SAMPLE_IMAGE = ROOT / 'imgs/260513_TC_Level/tiles/YOUR_IMAGE.tif'
MANUAL_MASK  = ROOT / 'masks_manual/YOUR_IMAGE_masks.tif'  # your manually downloaded mask

if SAMPLE_IMAGE.exists():
    img  = tifffile.imread(str(SAMPLE_IMAGE))
    auto_mask_path = ROOT / f'masks/260513_TC_Level/{SAMPLE_IMAGE.stem}_masks.tif'
    auto_mask = tifffile.imread(str(auto_mask_path))

    ncols = 3 if MANUAL_MASK.exists() else 2
    fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 5))

    axes[0].imshow(img if img.ndim == 3 else img, cmap='gray')
    axes[0].set_title('Original image')
    axes[0].axis('off')

    axes[1].imshow(auto_mask, cmap='tab20b')
    axes[1].set_title(f'Auto mask ({auto_mask.max()} cells)')
    axes[1].axis('off')

    if MANUAL_MASK.exists():
        manual_mask = tifffile.imread(str(MANUAL_MASK))
        axes[2].imshow(manual_mask, cmap='tab20b')
        axes[2].set_title(f'Manual mask ({manual_mask.max()} cells)')
        axes[2].axis('off')

    plt.tight_layout()
    plt.show()
else:
    print('Set SAMPLE_IMAGE and MANUAL_MASK paths above to run this cell.')